In [ ]:
import kagglehub

file_path = kagglehub.dataset_download("gomidor/conll-2003-english")

print("Path to dataset files:", file_path)

In [ ]:
!pip install evaluate
!pip install seqeval

In [ ]:
import os

print(file_path)
print(os.listdir(file_path))

In [ ]:
import os

# Use the file_path from the kagglehub download
base_path = file_path

train_file = os.path.join(base_path, "train.txt")
valid_file = os.path.join(base_path, "valid.txt")
test_file  = os.path.join(base_path, "test.txt")

# Removed: train_texts, train_labels = read_conll(train_file) as read_conll is not yet defined

In [ ]:
def read_conll(file_path):
    sentences = []
    labels = []

    words = []
    tags = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip() == "":
                sentences.append(words)
                labels.append(tags)
                words = []
                tags = []
            else:
                splits = line.split()
                words.append(splits[0])
                tags.append(splits[-1])

    return sentences, labels

train_texts, train_labels = read_conll(train_file)

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "tokens": train_texts,
    "ner_tags": train_labels
})

In [ ]:
label_list = list(set(tag for doc in train_labels for tag in doc))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [ ]:
def encode_labels(example):
    example["labels"] = [label2id[tag] for tag in example["ner_tags"]]
    return example

dataset = dataset.map(encode_labels)

Tokenization + Label Alignment

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_and_align_labels(example):
    tokenized_input = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_input.word_ids()

    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)  # special tokens
        elif word_idx != previous_word_idx:
            # Use the pre-encoded labels (integer IDs) directly from example["labels"]
            labels.append(example["labels"][word_idx])
        else:
            labels.append(-100)  # subword

        previous_word_idx = word_idx

    tokenized_input["labels"] = labels
    return tokenized_input

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)

Model Setup

In [ ]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label={i: l for i, l in enumerate(label_list)},
    label2id={l: i for i, l in enumerate(label_list)}
)

Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
from datasets import Dataset, DatasetDict # Import Dataset, DatasetDict

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    # logging_dir="./logs"
)

# --- Definition of full_tokenized_dataset moved here ---
# Read validation and test files
valid_texts, valid_labels = read_conll(valid_file)
test_texts, test_labels = read_conll(test_file)

# Create Dataset objects for validation and test
valid_dataset = Dataset.from_dict({
    "tokens": valid_texts,
    "ner_tags": valid_labels
})
test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "ner_tags": test_labels
})

# Encode labels for validation and test datasets
valid_dataset = valid_dataset.map(encode_labels)
test_dataset = test_dataset.map(encode_labels)

# Tokenize and align labels for validation and test datasets
tokenized_valid_dataset = valid_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=False)

# Create a DatasetDict with train, validation, and test splits
full_tokenized_dataset = DatasetDict({
    "train": tokenized_dataset,
    "validation": tokenized_valid_dataset,
    "test": tokenized_test_dataset
})

print("full_tokenized_dataset created successfully within the training cell!")
# --- End of full_tokenized_dataset definition ---

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_tokenized_dataset["train"],
    eval_dataset=full_tokenized_dataset["validation"],
    data_collator=data_collator
)

trainer.train()

In [ ]:
from datasets import Dataset, DatasetDict

# Read validation and test files
valid_texts, valid_labels = read_conll(valid_file)
test_texts, test_labels = read_conll(test_file)

# Create Dataset objects for validation and test
valid_dataset = Dataset.from_dict({
    "tokens": valid_texts,
    "ner_tags": valid_labels
})
test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "ner_tags": test_labels
})

# Encode labels for validation and test datasets
valid_dataset = valid_dataset.map(encode_labels)
test_dataset = test_dataset.map(encode_labels)

# Tokenize and align labels for validation and test datasets
tokenized_valid_dataset = valid_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=False)

# Create a DatasetDict with train, validation, and test splits
# 'tokenized_dataset' should be defined from an earlier step
full_tokenized_dataset = DatasetDict({
    "train": tokenized_dataset,
    "validation": tokenized_valid_dataset,
    "test": tokenized_test_dataset
})

print("full_tokenized_dataset created successfully!")

In [ ]:
from datasets import Dataset, DatasetDict

# Ensure base_path, train_file, valid_file, test_file are defined

# Read validation and test files
valid_texts, valid_labels = read_conll(valid_file)
test_texts, test_labels = read_conll(test_file)

# Create Dataset objects for validation and test
valid_dataset = Dataset.from_dict({
    "tokens": valid_texts,
    "ner_tags": valid_labels
})
test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "ner_tags": test_labels
})

# Encode labels for validation and test datasets
valid_dataset = valid_dataset.map(encode_labels)
test_dataset = test_dataset.map(encode_labels)

# Tokenize and align labels for validation and test datasets
tokenized_valid_dataset = valid_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=False)

# Create a DatasetDict with train, validation, and test splits
# 'tokenized_dataset' should be defined from an earlier step
full_tokenized_dataset = DatasetDict({
    "train": tokenized_dataset,
    "validation": tokenized_valid_dataset,
    "test": tokenized_test_dataset
})

print("full_tokenized_dataset created successfully!")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3

)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_tokenized_dataset["train"],
    eval_dataset=full_tokenized_dataset["validation"],
    data_collator=data_collator
    # tokenizer=tokenizer # Removed as it's an unexpected argument
)

trainer.train()

Evaluation

In [ ]:
import numpy as np
from evaluate import load

metric = load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

trainer.compute_metrics = compute_metrics

In [ ]:
from datasets import DatasetDict

# Read validation and test files
valid_texts, valid_labels = read_conll(valid_file)
test_texts, test_labels = read_conll(test_file)

# Create Dataset objects for validation and test
valid_dataset = Dataset.from_dict({
    "tokens": valid_texts,
    "ner_tags": valid_labels
})
test_dataset = Dataset.from_dict({
    "tokens": test_texts,
    "ner_tags": test_labels
})

# Encode labels for validation and test datasets
valid_dataset = valid_dataset.map(encode_labels)
test_dataset = test_dataset.map(encode_labels)

# Tokenize and align labels for validation and test datasets
tokenized_valid_dataset = valid_dataset.map(tokenize_and_align_labels, batched=False)
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=False)

# Create a DatasetDict with train, validation, and test splits
full_tokenized_dataset = DatasetDict({
    "train": tokenized_dataset,
    "validation": tokenized_valid_dataset,
    "test": tokenized_test_dataset
})

print(full_tokenized_dataset)

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    # logging_dir="./logs" # Removed to address deprecation warning
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=full_tokenized_dataset["train"],
    eval_dataset=full_tokenized_dataset["validation"],
    data_collator=data_collator
    # tokenizer=tokenizer # Removed as it's an unexpected argument
)

trainer.train()

Inference

In [ ]:
sentence = "John works at Google in California"

inputs = tokenizer(sentence, return_tensors="pt")

# Move input tensors to the same device as the model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model(**inputs)

predictions = outputs.logits.argmax(dim=2)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
predicted_labels = [id2label[p.item()] for p in predictions[0]] # Use id2label for mapping

for token, label in zip(tokens, predicted_labels):
    print(token, label)

Conclusion

POS tagging operates at the word level by assigning grammatical categories, whereas chunking operates at a higher level by grouping words into meaningful phrases. While POS tagging is simpler and focuses on individual tokens, chunking requires contextual understanding of sequences, making it moderately more complex.